# Notebook 3: Two-Zone Leptonic Model — MG3 J163554+3629

**Purpose:** Build and evaluate a two-zone leptonic model.

## Physical motivation
The one-zone model places all emission in a single compact region. A two-zone model separates:

- **Zone 1 (compact):** inside/near the BLR and Dust Torus → Sync + SSC + EC (BLR + Disk + DT). Dominates UV through gamma-ray.
- **Zone 2 (extended):** outside R_DT → Sync + SSC only (EC off, external photon fields too dilute at this distance). Contributes to radio and sub-mm.

**Shared parameters** (same AGN, same jet): z, θ, Γ, L_Disk, M_BH, η, T_DT, τ_DT, τ_BLR, R_in, R_ext

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import dill
%matplotlib inline

from jetset.jet_model import Jet
from jetset.data_loader import ObsData
from jetset.model_manager import FitModel
from jetset.template_2Dmodel import EBLAbsorptionTemplate

sed_data = ObsData.load('../../results/models/sed_data.pkl')
print('SED data loaded.')

---
## Part 1: Build Zone 1 (compact)
Zone 1 carries SSC + EC with BLR, Disk, DT. We load the Minuit best-fit parameters as starting values.

In [ ]:
zone1_jet = dill.load(open('../../results/models/minuit_model.pkl', 'rb')).jet_leptonic
zone1_jet.name = 'jet_compact'

# Apply refined Zone 1 parameters
zone1_jet.parameters.p.val           = 1.3
zone1_jet.parameters.p_1.val         = 2.9
zone1_jet.parameters.gamma_break.val = 400.0
zone1_jet.parameters.N.val           = 1000.0
zone1_jet.parameters.B.val           = 0.05
zone1_jet.parameters.gmin.val        = 2.0
zone1_jet.parameters.gmax.val        = 16000.0
zone1_jet.parameters.theta.val       = 3.0
zone1_jet.parameters.BulkFactor.val  = 15.0
zone1_jet.parameters.M_BH.val        = 1.9077e9
zone1_jet.parameters.L_Disk.val      = 1.5817e46
zone1_jet.parameters.accr_eff.val    = 0.083
zone1_jet.parameters.T_DT.val        = 600.9
zone1_jet.parameters.tau_DT.val      = 0.1
zone1_jet.parameters.tau_BLR.val     = 0.1
zone1_jet.parameters.R_inner_Sw.val  = 10.1
zone1_jet.parameters.R_ext_Sw.val    = 846.0
zone1_jet.parameters.R_H.val         = 1.5e18

zone1_jet.eval()
print(f'Zone 1: R_H = {zone1_jet.parameters.R_H.val:.3e} cm')
print(f'        R_DT = {zone1_jet.parameters.R_DT.val:.3e} cm')
print(f'        delta = {zone1_jet.get_beaming():.3f}')

---
## Part 2: Build Zone 2 (extended)
Zone 2 sits at 5 × R_DT. EC components are switched off — the external radiation fields are too dilute at this distance from the black hole.

In [ ]:
zone2_jet = Jet(
    name='jet_extended',
    electron_distribution='bkn',
    electron_distribution_log_values=False,
    beaming_expr='bulk_theta'
)

# Add EC structure for parameter linking — EC emission turned off below
zone2_jet.add_EC_component(
    EC_components_list=['EC_DT', 'EC_BLR', 'EC_Disk'],
    disk_type='MultiBB'
)

# Shared central engine
for par_name in ['z_cosm','L_Disk','M_BH','accr_eff','R_inner_Sw','R_ext_Sw',
                  'T_DT','tau_DT','tau_BLR','theta','BulkFactor']:
    zone2_jet.set_par(par_name, val=zone1_jet.parameters.get_par_by_name(par_name).val)

# Geometry: R = R_H * tan(theta_open)
zone2_jet.add_user_par(name='theta_open',
                        val=zone1_jet.parameters.theta_open.val, units='deg')
zone2_jet.make_dependent_par(
    par='R', depends_on=['R_H', 'theta_open'],
    par_expr='np.tan(np.radians(theta_open)) * R_H'
)

# Place Zone 2 outside R_DT
R_H2 = 5.0 * zone1_jet.parameters.R_DT.val
zone2_jet.set_par('R_H', val=R_H2)

# Electron distribution: same shape, lower N and B
zone2_jet.set_par('p',           val=zone1_jet.parameters.p.val)
zone2_jet.set_par('p_1',         val=zone1_jet.parameters.p_1.val)
zone2_jet.set_par('gamma_break', val=zone1_jet.parameters.gamma_break.val)
zone2_jet.set_par('gmin',        val=zone1_jet.parameters.gmin.val)
zone2_jet.set_par('gmax',        val=zone1_jet.parameters.gmax.val)
zone2_jet.set_par('B',           val=zone1_jet.parameters.B.val / 10.0)
zone2_jet.set_par('N',           val=zone1_jet.parameters.N.val * 1e-4)

# Turn OFF EC for Zone 2
zone2_jet.spectral_components.EC_DT.state  = 'off'
zone2_jet.spectral_components.EC_BLR.state = 'off'
zone2_jet.spectral_components.EC_Disk.state = 'off'
zone2_jet.spectral_components.DT.state     = 'off'
zone2_jet.spectral_components.Disk.state   = 'off'

zone2_jet.eval()
print(f'Zone 2: R_H = {zone2_jet.parameters.R_H.val:.3e} cm')
print(f'        B   = {zone2_jet.parameters.B.val:.4f} G')
print(f'        N   = {zone2_jet.parameters.N.val:.3e}')

---
## Part 3: Composite two-zone model with EBL

In [ ]:
two_zone_model = FitModel(jet=zone1_jet, name='TwoZone_Fit')
two_zone_model.add_component(zone2_jet)

ebl = EBLAbsorptionTemplate.from_name('Franceschini_2008')
two_zone_model.add_component(ebl)

two_zone_model.link_par(par_name='z_cosm',
                         from_model='Franceschini_2008', to_model='jet_compact')

shared_pars = ['z_cosm','theta','BulkFactor','theta_open',
               'L_Disk','M_BH','accr_eff','T_DT',
               'tau_DT','tau_BLR','R_inner_Sw','R_ext_Sw']
for par in shared_pars:
    try:
        two_zone_model.link_par(par_name=par,
                                 from_model='jet_extended', to_model='jet_compact')
    except Exception as e:
        print(f'  Could not link {par}: {e}')

two_zone_model.composite_expr = '(jet_compact + jet_extended) * Franceschini_2008'

two_zone_model.eval()
print('Two-zone model assembled.')
two_zone_model.show_model()

In [ ]:
# Plot full two-zone SED
p = two_zone_model.plot_model(sed_data=sed_data)
p.add_residual_plot(two_zone_model, sed_data)
p.setlim(y_min=1E-16, x_min=1E8)
plt.suptitle('MG3 J163554+3629 — Two-zone model (prefit)', fontsize=12)
plt.savefig('../../results/plots/twozone_prefit_sed.png', dpi=150, bbox_inches='tight')
plt.show()

two_zone_model.save_model('../../results/models/twozone_model.pkl')
print('Saved: twozone_model.pkl')

In [ ]:
# Zone contribution plot
fig, ax = plt.subplots(figsize=(10, 6))

for jet, label, color, ls in [
    (zone1_jet, 'Zone 1 (compact: EC+SSC+Sync)', 'steelblue', '--'),
    (zone2_jet, 'Zone 2 (extended: SSC+Sync)',    'darkorange','--'),
]:
    jet.eval()
    nu = jet.SED.nu.value
    fl = jet.SED.nuFnu.value
    g  = np.isfinite(fl) & np.isfinite(nu) & (nu > 0) & (fl > 0)
    ax.loglog(nu[g], fl[g], color=color, lw=1.5, ls=ls, alpha=0.8, label=label)

nu_tot = two_zone_model.SED.nu.value
fl_tot = two_zone_model.SED.nuFnu.value
g_tot  = np.isfinite(fl_tot) & np.isfinite(nu_tot) & (nu_tot > 0) & (fl_tot > 0)
ax.loglog(nu_tot[g_tot], fl_tot[g_tot], 'k-', lw=2, label='Total (+ EBL)')

mask_ul  = sed_data.table['UL'].astype(bool)
mask_det = ~mask_ul
ax.errorbar(sed_data.table['nu_data'][mask_det],
            sed_data.table['nuFnu_data'][mask_det],
            yerr=sed_data.table['dnuFnu_data'][mask_det],
            fmt='o', color='gray', ms=4, label='data', zorder=5)
ax.scatter(sed_data.table['nu_data'][mask_ul],
           sed_data.table['nuFnu_data'][mask_ul],
           marker='v', color='gray', s=30, zorder=5)

ax.set_xlim(1e8, 1e27); ax.set_ylim(1e-16, 1e-10)
ax.set_xlabel(r'$\nu$ (Hz)', fontsize=13)
ax.set_ylabel(r'$\nu F_\nu$ (erg cm$^{-2}$ s$^{-1}$)', fontsize=13)
ax.set_title('MG3 J163554+3629 — Zone contributions', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('../../results/plots/twozone_contributions.png', dpi=150, bbox_inches='tight')
plt.show()